In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_'
login(token=HF_TOKEN)

In [4]:
# scripts/02_emotion_direction_extraction/analyze_qwen_architecture.py
"""
Qwen Model Architecture Analysis Script
Analyzes the structure of Qwen3-4B-Instruct-2507 to understand:
- Layer structure and naming
- Attention and MLP module locations
- Hidden state access patterns
- Residual connection points
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import json

# Model configuration
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DEVICE = "cpu"  # CPU is sufficient for architecture inspection
DTYPE = torch.float32

print("=" * 80)
print(f"Analyzing Qwen Model: {MODEL_NAME}")
print("=" * 80)

# Load model and tokenizer
print("\n[1] Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True
)
model.eval()

print("\n[2] Basic Model Information")
print("-" * 40)
print(f"Model type: {type(model)}")
print(f"Model class: {model.__class__.__name__}")
print(f"Model config: {model.config.__class__.__name__}")
print(f"Hidden size: {getattr(model.config, 'hidden_size', 'N/A')}")
print(f"Num layers: {getattr(model.config, 'num_hidden_layers', 'N/A')}")
print(f"Num attention heads: {getattr(model.config, 'num_attention_heads', 'N/A')}")
print(f"Vocab size: {getattr(model.config, 'vocab_size', 'N/A')}")

print("\n[3] Top-level Model Structure")
print("-" * 40)
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")
    if hasattr(module, 'config'):
        print(f"    config: {module.config.__class__.__name__}")

# Explore the model structure recursively to find layers
print("\n[4] Exploring Model Structure for Layer Access")
print("-" * 40)

# Common layer access patterns in different model architectures
layer_access_patterns = [
    "model.layers",  # Llama, Mistral, many others
    "transformer.h",  # GPT-2 style
    "model.decoder.layers",  # T5 style
    "encoder.layers",  # Encoder-only models
    "model.model.layers",  # Some nested structures
    "model.transformer.blocks",  # Some vision models
    "model.layers",  # Qwen often uses this
    "qwen.model.layers",  # Explicit Qwen path
]

# Try different access patterns to find layers
layers = None
found_pattern = None

for pattern in layer_access_patterns:
    try:
        # Split pattern by dots and navigate
        attrs = pattern.split('.')
        current = model
        valid = True
        for attr in attrs:
            if hasattr(current, attr):
                current = getattr(current, attr)
            else:
                valid = False
                break
        
        if valid and hasattr(current, '__len__') and len(current) > 0:
            layers = current
            found_pattern = pattern
            print(f"✓ Found layers via: model.{pattern}")
            print(f"  Number of layers: {len(layers)}")
            print(f"  Layer type: {type(layers[0]).__name__}")
            break
    except:
        continue

if not layers:
    print("✗ Could not find layers via common patterns. Trying deep search...")
    
    # Deep recursive search for any attribute that looks like a layer list
    def find_layer_list(module, path="", depth=0, max_depth=5):
        if depth > max_depth:
            return None
        
        results = []
        for name, submodule in module.named_children():
            current_path = f"{path}.{name}" if path else name
            
            # Check if this is a list/tuple of modules that could be layers
            if hasattr(submodule, '__len__') and len(submodule) > 0:
                first = submodule[0]
                if hasattr(first, '__class__') and 'layer' in first.__class__.__name__.lower():
                    results.append((current_path, submodule))
            
            # Recurse
            deeper = find_layer_list(submodule, current_path, depth + 1, max_depth)
            if deeper:
                results.extend(deeper)
        
        return results
    
    layer_candidates = find_layer_list(model)
    if layer_candidates:
        for path, layer_list in layer_candidates:
            print(f"  Found candidate: {path} with {len(layer_list)} layers")
            if len(layer_list) == getattr(model.config, 'num_hidden_layers', 0):
                print(f"  ✓ This matches config's num_hidden_layers!")
                layers = layer_list
                found_pattern = path
                break

print("\n[5] Layer Structure Analysis")
print("-" * 40)

if layers:
    first_layer = layers[0]
    print(f"First layer type: {type(first_layer).__name__}")
    
    print("\nFirst layer components:")
    for name, module in first_layer.named_children():
        print(f"  {name}: {type(module).__name__}")
        
        # For attention submodules, show their structure
        if 'attn' in name.lower():
            print(f"    Attention submodules:")
            for subname, submodule in module.named_children():
                print(f"      {subname}: {type(submodule).__name__}")
        
        # For MLP submodules, show their structure
        if 'mlp' in name.lower() or 'ffn' in name.lower():
            print(f"    MLP submodules:")
            for subname, submodule in module.named_children():
                print(f"      {subname}: {type(submodule).__name__}")
    
    # Analyze attention output pattern
    print("\n[6] Attention Output Structure Analysis")
    print("-" * 40)
    
    # Check how attention outputs are structured
    if hasattr(first_layer, 'self_attn') or hasattr(first_layer, 'attention'):
        attn_module = getattr(first_layer, 'self_attn', None) or getattr(first_layer, 'attention', None)
        
        # Try to understand the forward output structure
        print("To analyze attention output structure, we need to run a forward pass...")
        
        # Create a simple input
        test_input = tokenizer("Hello, how are you?", return_tensors="pt")
        
        # Register hook to see attention output structure
        attention_outputs = {}
        
        def attention_hook(module, input, output):
            print(f"  Attention hook triggered!")
            print(f"  Input type: {type(input)}")
            print(f"  Input length: {len(input) if isinstance(input, tuple) else 'N/A'}")
            print(f"  Output type: {type(output)}")
            print(f"  Output length: {len(output) if isinstance(output, tuple) else 'N/A'}")
            
            if isinstance(output, tuple):
                for i, out in enumerate(output):
                    if isinstance(out, torch.Tensor):
                        print(f"    output[{i}] shape: {out.shape}")
                        attention_outputs['attention_output'] = out
                    else:
                        print(f"    output[{i}] type: {type(out)}")
            elif isinstance(output, torch.Tensor):
                print(f"  Output shape: {output.shape}")
                attention_outputs['attention_output'] = output
        
        # Register hook
        hook = attn_module.register_forward_hook(attention_hook)
        
        # Run forward pass
        with torch.no_grad():
            print("\nRunning forward pass with hook...")
            outputs = model(**test_input, output_hidden_states=True)
        
        hook.remove()
        
        # Analyze hidden states
        print("\n[7] Hidden States Analysis")
        print("-" * 40)
        if hasattr(outputs, 'hidden_states'):
            print(f"Hidden states available: {len(outputs.hidden_states)} tensors")
            for i, hs in enumerate(outputs.hidden_states):
                print(f"  hidden_states[{i}] shape: {hs.shape}")
            
            # Understand what each hidden state represents
            print("\nHidden state interpretation:")
            print(f"  hidden_states[0]: Usually embedding layer output")
            print(f"  hidden_states[1]: After layer 0")
            print(f"  hidden_states[2]: After layer 1")
            print(f"  ...")
            print(f"  hidden_states[-1]: Final layer output (before LM head)")
        else:
            print("No hidden_states in outputs. Check if output_hidden_states=True is respected.")
        
        # Check if model has lm_head
        print("\n[8] LM Head Analysis")
        print("-" * 40)
        if hasattr(model, 'lm_head'):
            print(f"lm_head type: {type(model.lm_head).__name__}")
            if hasattr(model.lm_head, 'weight'):
                print(f"lm_head weight shape: {model.lm_head.weight.shape}")
        elif hasattr(model, 'model') and hasattr(model.model, 'lm_head'):
            print(f"model.lm_head type: {type(model.model.lm_head).__name__}")
        else:
            print("No separate lm_head found - might be tied to embeddings")
            
        # Check embedding layer
        print("\n[9] Embedding Layer Analysis")
        print("-" * 40)
        if hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
            print(f"embed_tokens found in model.model")
        elif hasattr(model, 'transformer') and hasattr(model.transformer, 'wte'):
            print(f"wte (word token embeddings) found")
        else:
            # Search for embeddings
            for name, module in model.named_modules():
                if 'embed' in name.lower() and hasattr(module, 'weight'):
                    print(f"Found embeddings at: {name}")
                    print(f"  Shape: {module.weight.shape}")
                    break
else:
    print("Could not locate layers for detailed analysis")

print("\n[10] Model Configuration Summary")
print("-" * 40)
config_dict = model.config.to_dict()
important_keys = [
    'architectures', 'model_type', 'hidden_size', 'num_hidden_layers',
    'num_attention_heads', 'intermediate_size', 'vocab_size',
    'max_position_embeddings', 'rms_norm_eps', 'rope_theta'
]

for key in important_keys:
    if key in config_dict:
        print(f"{key}: {config_dict[key]}")

# Check for Qwen-specific configurations
print("\n[11] Qwen-Specific Attributes")
print("-" * 40)
qwen_specific = ['rope_scaling', 'yarn_config', 'sliding_window', 'attention_dropout']
for attr in qwen_specific:
    if hasattr(model.config, attr):
        value = getattr(model.config, attr)
        print(f"{attr}: {value}")

print("\n[12] Recommended Access Patterns for Code Adaptation")
print("-" * 40)

if found_pattern:
    print(f"✓ Layer access: model.{found_pattern}[layer_idx]")
    
    # Check attention module path
    if hasattr(first_layer, 'self_attn'):
        print(f"✓ Attention module: model.{found_pattern}[layer_idx].self_attn")
        print(f"  - Attention output is likely output[0] or output depending on implementation")
    elif hasattr(first_layer, 'attention'):
        print(f"✓ Attention module: model.{found_pattern}[layer_idx].attention")
        print(f"  - Check attention output structure with hook before implementing")
    
    # Check MLP module path
    if hasattr(first_layer, 'mlp'):
        print(f"✓ MLP module: model.{found_pattern}[layer_idx].mlp")
    elif hasattr(first_layer, 'ffn'):
        print(f"✓ MLP module: model.{found_pattern}[layer_idx].ffn")
    
    # Check layer norm paths
    if hasattr(first_layer, 'input_layernorm'):
        print(f"✓ Input layernorm: model.{found_pattern}[layer_idx].input_layernorm")
    if hasattr(first_layer, 'post_attention_layernorm'):
        print(f"✓ Post-attention layernorm: model.{found_pattern}[layer_idx].post_attention_layernorm")
    
    # Hidden states pattern
    print(f"✓ Hidden states: outputs.hidden_states[layer_idx + 1] for layer output")
    print(f"  - hidden_states[0]: embedding output")
    print(f"  - hidden_states[1]: after layer 0")
    print(f"  - hidden_states[-1]: final output")

print("\n" + "=" * 80)
print("Analysis complete. Use this information to update the emotion extraction code.")
print("=" * 80)

Analyzing Qwen Model: Qwen/Qwen3-4B-Instruct-2507

[1] Loading model and tokenizer...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


[2] Basic Model Information
----------------------------------------
Model type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
Model class: Qwen3ForCausalLM
Model config: Qwen3Config
Hidden size: 2560
Num layers: 36
Num attention heads: 32
Vocab size: 151936

[3] Top-level Model Structure
----------------------------------------
  model: Qwen3Model
    config: Qwen3Config
  lm_head: Linear

[4] Exploring Model Structure for Layer Access
----------------------------------------
✓ Found layers via: model.model.layers
  Number of layers: 36
  Layer type: Qwen3DecoderLayer

[5] Layer Structure Analysis
----------------------------------------
First layer type: Qwen3DecoderLayer

First layer components:
  self_attn: Qwen3Attention
    Attention submodules:
      q_proj: Linear
      k_proj: Linear
      v_proj: Linear
      o_proj: Linear
      q_norm: Qwen3RMSNorm
      k_norm: Qwen3RMSNorm
  mlp: Qwen3MLP
    MLP submodules:
      gate_proj: Linear
      up_proj: Li